# Chemistry-informed scoring of the Track A designs

A soft, rank-only score that sits **downstream of ipSAE** and separates designs that tie on ipSAE. The HER2 epitope the
trastuzumab scFv binds is acidic (five Asp/Glu in the Boltz poses: E558, D560, D570, D596, E598), so a paratope that offers a
positive side chain near one of those carboxylates should bind harder. The score has two terms (`docs/chem_score_spec.md`):

- **contact** - a short-range salt-bridge reward: for each substitution that introduces Lys or Arg, 1.0 when its charged N is
  within 3.5 A of an epitope carboxylate O, falling linearly to 0 at 4.0 A (the 2.5 + 1.5 A reach on the whiteboard).
- **coulomb** - a long-range steering term: the screened-Coulomb energy of the mutated charge sites against the antigen,
  parent minus design, so a more favourable configuration is larger. This keeps H:D31K/R (which steer from 5-8 A, too far for a
  salt bridge) in play.

A **pose filter** drops Boltz seeds whose epitope does not overlap the parent's by at least 0.5, so an off-site pose cannot
dominate. A design with no on-site seed gets NaN and is left to ipSAE. The score **never vetoes**; it only re-ranks.

The geometry is `charge_contact_report` in `design_pipeline.scoring` (biotite), the single source of truth this notebook calls.

## Parameters

Paths to the Boltz-2 run on the GPU node. Chain A of each model is the scFv, chain B is the HER2 fragment (480-607).

In [1]:
RUN = "/scratch/foertsch/protecher_track_a_run"
PARENT = "trastuzumab_scfv"
SEEDS = (0, 1)
CRYSTAL = "/scratch/foertsch/protecher_cache/structures/1N8Z.cif"

## Setup

In [2]:
import glob
import statistics

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

from design_pipeline.scoring import charge_contact_report


def find_cif(name, seed):
    """The model_0 CIF of one complex at one seed, under fold_a or fold_b."""
    hits = [h for b in "ab" for h in glob.glob(
        f"{RUN}/fold_{b}/predictions/seed{seed}/*/{name}/boltz_results_{name}/predictions/{name}/{name}_model_0.cif")]
    if len(hits) != 1:
        raise FileNotFoundError(f"{name} seed{seed}: {hits}")
    return hits[0]


designs = pd.read_csv(f"{RUN}/design/designs_final.csv")
len(designs)

14

## The epitope is acidic

The premise, read off the parent's Boltz pose: which epitope residues are Asp or Glu.

In [3]:
import biotite.structure as struc
import biotite.structure.io.pdbx as pdbx

a = pdbx.get_structure(pdbx.CIFFile.read(find_cif(PARENT, 0)), model=1)
a = a[a.element != "H"]
scfv, antigen = a[a.chain_id == "A"], a[a.chain_id == "B"]
d = np.linalg.norm(scfv.coord[:, None, :] - antigen.coord[None, :, :], axis=-1)
epitope = sorted(set(antigen.res_id[(d <= 5.0).any(axis=0)].tolist()))
ca = antigen[antigen.atom_name == "CA"]
name = {int(i): n for i, n in zip(ca.res_id, ca.res_name)}
acids = [f"{name[r]} {r + 479}" for r in epitope if name.get(r) in ("ASP", "GLU")]
print(f"{len(epitope)} epitope residues, {len(acids)} acidic:")
acids

27 epitope residues, 6 acidic:


['GLU 558', 'ASP 560', 'ASP 570', 'ASP 585', 'ASP 596', 'GLU 598']

## Score every design pose

Each design, both seeds, against the parent's prediction of the same seed.

In [4]:
rows = []
for _, dsg in designs.iterrows():
    for s in SEEDS:
        rep = charge_contact_report(find_cif(dsg["design"], s), find_cif(PARENT, s))
        rep.update(design=dsg["design"], seed=s, modules=dsg["modules"])
        rows.append(rep)
poses = pd.DataFrame(rows)
poses[["design", "seed", "eligible", "contact", "coulomb", "epitope_overlap", "n_added_cation", "paratope_net_charge"]].round(3)

       design  seed  eligible  contact  coulomb  epitope_overlap  n_added_cation  paratope_net_charge
0   design_01     0       0.0    0.000    0.099            0.256             1.0                  4.0
1   design_01     1       1.0    0.000    0.187            0.893             1.0                  4.0
2   design_02     0       0.0    0.000    0.207            0.231             2.0                  3.0
3   design_02     1       1.0    1.000    0.500            0.857             2.0                  6.0
4   design_03     0       0.0    0.000    0.253            0.250             2.0                  6.0
5   design_03     1       0.0    0.000    0.137            0.000             2.0                  3.0
6   design_04     0       0.0    0.000    0.283            0.270             3.0                  5.0
7   design_04     1       1.0    0.000    0.359            0.778             3.0                  5.0
8   design_05     0       0.0    0.000    0.371            0.324             4.0  

## Aggregate over eligible seeds and re-rank

Average the two terms over the seeds that pass the pose filter, z-score across the pool, and rank by `chem = z(contact) + z(coulomb)`.
Designs with no on-site seed are `NaN` (left to ipSAE).

In [5]:
def aggregate(g):
    elig = g[g.eligible >= 1.0]
    return pd.Series({
        "n_elig": len(elig),
        "contact": elig.contact.mean() if len(elig) else np.nan,
        "coulomb": elig.coulomb.mean() if len(elig) else np.nan,
        "para_net": g.paratope_net_charge.mean(),
        "modules": g.modules.iloc[0],
    })


def zscore(s):
    sd = s.std(ddof=0)
    return (s - s.mean()) / (sd if sd else 1.0)


by = poses.groupby("design", sort=False).apply(aggregate, include_groups=False)
on_site = by[by.n_elig > 0]
by["chem"] = np.nan
by.loc[on_site.index, "chem"] = zscore(on_site.contact) + zscore(on_site.coulomb)
ranked = by.sort_values("chem", ascending=False)
ranked[["n_elig", "contact", "coulomb", "para_net", "chem", "modules"]].round(3)

           n_elig  contact  coulomb  para_net   chem                                            modules
design                                                                                                 
design_05       1      1.0    0.759       5.0  2.638            ADAPT H:D31K + ADAPT L:D28K + Moon AH16
design_13       1      1.0    0.568       5.5  1.583  ADAPT H:D31K + ADAPT L:D28K + Moon AH16 + Ng V...
design_07       1      1.0    0.564       4.5  1.561            ADAPT H:D31R + ADAPT L:D28K + Moon AH06
design_02       1      1.0    0.500       4.5  1.213            ADAPT H:D31K + ADAPT L:D28K + Moon AH06
design_06       2      1.0    0.482       3.5  1.110  ADAPT H:D31K + ADAPT L:D28K + Moon Modified AH...
design_11       2      1.0    0.480       6.0  1.100  ADAPT H:D31K + ADAPT L:D28K + Moon AH06 + Ng V...
design_14       2      1.0    0.358       3.0  0.431  ADAPT L:D28K + Moon Modified AH06 W98D + Ng VL...
design_04       1      0.0    0.359       5.0 -1.642            

## Reading the result

The score separates the ipSAE-parity designs by mechanism:

- **Salt bridge.** Every design that carries **L:D28K** rises to the top with a full contact: its lysine reaches the E598
  carboxylate at 2-3 A. D28 itself is an aspartate 4.5 A from E598 in the crystal (acid against acid), so the D-to-K swap turns a
  clash into a bridge.
- **Steering only.** Designs whose only added cation is **H:D31K** or **H:D31R** score zero contact (the side chain sits 5-8 A
  from the nearest carboxylate) but keep a positive coulomb term, so they are ranked below the bridge designs rather than discarded.
- **Flagged.** The Moon AH16 designs fold off-site (epitope overlap ~0.1), so they have no eligible seed and are left to ipSAE,
  which already rejects them.

`paratope_net_charge` is reported, not gated: a large positive paratope is a developability flag (general knowledge, not checked here).

Two numbers here come from model knowledge, not a source checked in this run: the 3.5-4.0 A salt-bridge ramp and histidine as a
weak cation (weight 0.5 in the scorer).